# Tái hiện SEEK (Sousa23) và KGsim2vec (Sousa24) cho dự đoán PPI

Notebook này chạy lại **toàn bộ pipeline gốc** của hai repo tác giả, rồi so từng con số với bài báo.

| Phần | Repo | Kiểm tra |
|---|---|---|
| A. KGsim2vec | `liseda-lab/ExplainablePPI` | A3 chọn aspect → A4 SML Java gốc tính Resnik-max IC Seco → A5 bảng 1/10 Sousa24 (DT, DT6, RF, XGB) → A6 size & informativeness của giải thích (Fig. 4) |
| B. SEEK | `liseda-lab/seek` | B3 dựng lại KG → B4 embedding lớp RDF2Vec → B5 biểu diễn cặp DCA → B6 bảng 2 Sousa23 (RF, XGB, MLP) → B7 bảng 3, 4 (necessary / sufficient) |
| C. Tổng hợp | | Một bảng so sánh với bài báo, nén kết quả để tải về |

**Chính xác hay chỉ tương đương?**
- *Chính xác từng số*: aspect (A3), đặc trưng SML (A4), biểu diễn cặp SEEK khi dùng embedding của tác giả (B5). Đã kiểm trước: lệch 0.
- *Tương đương thống kê*: mọi mô hình học máy. Repo gốc không cố định `random_state`; notebook đặt `random_state=0` để chạy lại ra cùng số, nên F1 có thể lệch vài phần nghìn so với bài báo.
- *Ngẫu nhiên*: huấn luyện lại RDF2Vec (`SEEK_EMB = 'retrain'`), vì random walk và Word2Vec không cố định seed trong repo.

**Hai lưu ý quan trọng khi đọc kết quả**
1. Repo KGsim2vec cho **52** aspect, gồm `protein-containing complex` (GO_0032991). Bài báo nói đã bỏ nhánh này để tránh vòng tròn, và bảng 10 ghi 51 aspect. Vì vậy A5 báo cả hai cấu hình 52 và 51.
2. Repo KGsim2vec có hai lỗi nhỏ khi chạy trên Linux/Colab: classpath dùng `;` (của Windows), và file `Run1.txt` bị cắt còn 8 dòng. Notebook dùng `:` và `Run1(complete).txt`. Ngoài ra repo viết cho scikit-learn 0.21 (Python 3.7), không cài được trên Colab. A6 vá hai chỗ tương thích mà không đổi thuật toán: `export_text` nay bắt buộc tham số tên, và `sklearn.externals.six/joblib` đã bị gỡ.

Chạy bằng **Runtime → Run all**. Không cần restart: phần SEEK chạy trong một virtualenv riêng với đúng phiên bản thư viện của repo.

In [ ]:
# ===== Cấu hình =====
RUN_KGSIM2VEC = True
RUN_SEEK = True

KG_LIME_PAIRS = 200        # số cặp test fold 1 để chạy LIME3/LIME8 (repo gốc chạy cho mọi cặp test, khoảng 1 giờ mỗi mô hình)
SEEK_REBUILD_KG = True     # B3: dựng lại KG từ go.owl + GAF và so với KG.gpickle của tác giả
SEEK_EMB = 'authors'       # 'authors' = embedding lớp của tác giả (tái hiện chính xác) | 'retrain' = chạy lại RDF2Vec
SEEK_PAIR_MODE = 'fast'   # 'fast' = cùng định nghĩa DCA, có cache (vài phút) + đối chiếu với hàm repo trên 500 cặp | 'repo' = gọi run_PairEmbeddings cho mọi cặp (1–2 giờ)
SEEK_ALGS = ['RF', 'XGB', 'MLP']
SEEK_EXPL_MAX_PER_FOLD = None   # None = toàn bộ 47,142 cặp như bài báo (~20 phút mỗi mô hình); đặt 300 để chạy nhanh (lấy mẫu ngẫu nhiên mỗi fold)

import os, json, subprocess, time
W = '/content'; S = W + '/repro_scripts'; RES = W + '/repro_results'
os.makedirs(S, exist_ok=True); os.makedirs(RES, exist_ok=True)

## Script dùng chung
Bốn script dưới đây được ghi ra `/content/repro_scripts`. Chúng gọi trực tiếp hàm trong repo gốc, chỉ thêm phần đọc kết quả và so với bài báo.

In [ ]:
%%writefile /content/repro_scripts/kg_compare_sml.py
"""So file SML vua chay voi file tac gia commit (Experimental_Results/Results/), can cot theo URL aspect.
python kg_compare_sml.py <new_ss> <new_aspect_urls> <authors_ss> <authors_sas>"""
import sys, numpy as np
def load(p):
    d = {}
    for l in open(p):
        t = l.rstrip('\n').split('\t'); d[(t[0].split('/')[-1], t[1].split('/')[-1])] = np.array(t[2:], float)
    return d
new, old = load(sys.argv[1]), load(sys.argv[3])
un = [l.split('\t')[0].strip() for l in open(sys.argv[2]) if l.strip()]
uo = [l.split('\t')[0].strip() for l in open(sys.argv[4]) if l.strip()]
print('so aspect: moi', len(un), '| tac gia', len(uo), '| cung tap aspect:', set(un) == set(uo))
perm = [un.index(u) for u in uo]
ks = [k for k in old if k in new]
A = np.array([new[k][perm] for k in ks]); B = np.array([old[k] for k in ks])
d = np.abs(A - B)
print('so cap: moi', len(new), '| tac gia', len(old), '| chung', len(ks))
print('ty le o lech > 1e-9: %.6f | lech lon nhat: %.3g' % ((d > 1e-9).mean(), d.max()))

In [ ]:
%%writefile /content/repro_scripts/kg_predict.py
"""Tai hien bang 1 / bang 10 Sousa24: 10-fold, DT/DT6/RF/XGB (tham so mac dinh nhu Prediction/ML.py).
Dung: python kg_predict.py <repo_dir> <ss_file> <aspect_file> <out_json> [algs] [n_folds]
aspect_file: moi dong 'url<TAB>ten', cung thu tu cot voi ss_file."""
import sys, json, time, numpy as np
from statistics import median
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import xgboost as xgb
repo, ss_file, asp_file, out = sys.argv[1:5]
algs = sys.argv[5].split(',') if len(sys.argv) > 5 else ['DT6', 'DT', 'RF', 'XGB']
nf = int(sys.argv[6]) if len(sys.argv) > 6 else 10
D = repo + '/Experimental_Results/Data/'
pairs, y = [], []
for l in open(D + 'v11(score950).txt'):
    t = l.rstrip('\n').split('\t'); pairs.append((t[0], t[1])); y.append(int(t[2]))
y = np.array(y)
ss = {}
for l in open(ss_file):
    t = l.rstrip('\n').split('\t'); ss[(t[0].split('/')[-1], t[1].split('/')[-1])] = [float(v) for v in t[2:]]
X = np.array([ss[p] for p in pairs])
asp = [l.split('\t')[0].strip() for l in open(asp_file) if l.strip()]
assert len(asp) == X.shape[1], (len(asp), X.shape)
CX = 'http://purl.obolibrary.org/obo/GO_0032991'
keep51 = [i for i, a in enumerate(asp) if a != CX]
def fold_idx(r):
    f = D + 'StratifiedPartitions/Indexes__crossvalidationTest__Run' + ('1(complete)' if r == 1 else str(r)) + '.txt'
    return [int(l) for l in open(f) if l.strip()]
def model(a, seed):
    if a == 'DT': return DecisionTreeClassifier(random_state=seed)
    if a == 'DT6': return DecisionTreeClassifier(max_depth=6, random_state=seed)
    if a == 'RF': return RandomForestClassifier(random_state=seed, n_jobs=-1)
    if a == 'XGB': return xgb.XGBClassifier(random_state=seed, n_jobs=-1)
res = {}
for cfg, cols in [('52_aspects_repo', list(range(X.shape[1]))), ('51_aspects_no_complex', keep51)]:
    for a in algs:
        w = []
        for r in range(1, nf + 1):
            te = fold_idx(r); m = np.ones(len(y), bool); m[te] = False
            clf = model(a, 0).fit(X[m][:, cols], y[m])
            w.append(f1_score(y[te], clf.predict(X[te][:, cols]), average='weighted'))
        q1, q3 = np.percentile(w, [25, 75])
        res[cfg + '/' + a] = {'median_WAF': round(median(w), 4), 'IQR': round(q3 - q1, 4), 'folds': [round(v, 4) for v in w]}
        print(cfg, a, res[cfg + '/' + a]['median_WAF'], 'IQR', res[cfg + '/' + a]['IQR'], flush=True)
json.dump(res, open(out, 'w'), indent=1)

In [ ]:
%%writefile /content/repro_scripts/kg_explain.py
"""Giai thich + danh gia size/informativeness cua KGsim2vec (Sousa24, Fig. 4) tren fold 1, dung code goc cua repo.
Chay trong thu muc repo ExplainablePPI: python kg_explain.py <ss_file> <names_file> <N_LIME> <out_dir>
names_file: ten aspect (dang get_semantic_aspects tra ve), moi dong mot ten, dung thu tu cot cua ss_file."""
import sys, os, json, time, statistics, numpy as np
sys.path.insert(0, '.')
# gplearn_variations (vendored, viet cho sklearn 0.21) con import sklearn.externals.six/joblib da bi go khoi sklearn moi
import six, joblib, sklearn.externals
sklearn.externals.six = six; sys.modules['sklearn.externals.six'] = six
sklearn.externals.joblib = joblib; sys.modules['sklearn.externals.joblib'] = joblib
import supervised_learning as SL, evaluation_explanations as EV
from Prediction import ML, LIMEexplanations
from sklearn.tree import export_text as _et
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
ML.export_text = lambda clf, names: _et(clf, feature_names=list(names))  # sklearn moi: feature_names la keyword-only
ss_file, names_file, N, out = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
os.makedirs(out, exist_ok=True)
D = 'Experimental_Results/Data/'
aspects = [l.rstrip('\n') for l in open(names_file) if l.strip()]
labels, pairs = SL.process_dataset(D + 'v11(score950).txt')
dict_SS = SL.process_SS_file(ss_file)
X = np.array([dict_SS[(a, b)] for a, b in pairs]); y = np.array(labels); pairs = np.array(pairs)
te = SL.process_indexes_partition(D + 'StratifiedPartitions/Indexes__crossvalidationTest__Run1(complete).txt')
tr = sorted(set(range(len(y))) - set(te))
Xtr, Xte, ytr, yte = list(X[tr]), list(X[te]), list(y[tr]), list(y[te]); pte = list(pairs[te])
dic_ics = EV.process_IC(D + 'ICSeco_GOterms.txt'); n2u = EV.GOurl2GOname(D + 'go-basic.owl', aspects)
res = {}; t0 = time.time()
for alg, d in [('DT6', 6), ('DT', None)]:
    f = out + f'/{alg}_'
    ML.performance_DT(dict_SS, pte, Xtr, Xte, ytr, yte, aspects, f + 'pred.txt', f + 'model.txt', f + 'eval.txt', f + 'expl.txt', d=d)
    EV.process_models(f + 'expl.txt', f + 'evalexpl.txt', dic_ics, n2u, aspects)
    last = open(f + 'evalexpl.txt').read().strip().split('\n')[-1].split('\t')
    res[alg] = {'median_SumIC': float(last[2]), 'median_AvgIC': float(last[3]), 'median_size': float(last[4]),
                'WAF_fold1': float(open(f + 'eval.txt').readline().split('\t')[1])}
    print(alg, res[alg], round(time.time() - t0), 's', flush=True)
rng = np.random.RandomState(0); idx = sorted(rng.choice(len(yte), min(N, len(yte)), replace=False))
for alg, clf in [('RF', RandomForestClassifier(random_state=0)), ('XGB', xgb.XGBClassifier(random_state=0))]:
    clf.fit(np.array(Xtr), np.array(ytr))
    s = {3: [], 8: []}
    for i in idx:
        for k in (3, 8):
            rule = LIMEexplanations.LIME_explanation(Xtr, Xte, i, clf, aspects, k)
            s[k].append(EV.process_rule(rule, aspects, dic_ics, n2u))
    res[alg] = {f'LIME{k}': {'median_SumIC': statistics.median(v[0] for v in s[k]), 'median_AvgIC': statistics.median(v[1] for v in s[k]),
                             'median_size': statistics.median(v[2] for v in s[k])} for k in (3, 8)}
    res[alg]['n_pairs'] = len(idx)
    print(alg, res[alg], round(time.time() - t0), 's', flush=True)
json.dump(res, open(out + '/kg_explain_summary.json', 'w'), indent=1)

In [ ]:
%%writefile /content/repro_scripts/seek_repro.py
"""Tai hien SEEK (Sousa23) cho PPI. Chay trong thu muc goc cua repo seek.
python seek_repro.py STAGE [options]
  kg                      : dung lai KG tu go.owl + GAF bang build_kg.run_save_graph, so voi KG.gpickle tac gia
  emb  retrain|authors    : authors = dung embedding lop cua tac gia; retrain = chay lai RDF2Vec (walker + Word2Vec)
  pairs EMB_FILE OUT      : tinh bieu dien cap bang generate_pair_representations.run_PairEmbeddings, so voi file tac gia
  ml   PAIR_FILE TAG [ALGS] : 10-fold RF/XGB/MLP (tham so mac dinh nhu train_ml.py), so voi Table 2
  expl PAIR_FILE EMB_FILE TAG ALG [MAX_PER_FOLD] : necessary/sufficient (Table 3, Table 4)"""
import sys, os, json, time, pickle, ast
import numpy as np
P = './PPI/'
DATA = P + 'STRING_pairs.txt'
PART = P + 'StratifiedPartitions/Indexes__crossvalidationTest__Run'
PAPER_T2 = {'XGB': (0.920, 0.910, 0.915), 'RF': (0.922, 0.892, 0.910), 'MLP': (0.908, 0.924, 0.917)}
PAPER_T3 = {'MLP': (-0.157, -0.137, -0.148, 0.932, 0.959, 0.950), 'XGB': (-0.109, -0.120, -0.113, 1.0, 1.0, 1.0),
            'RF': (-0.106, -0.153, -0.125, 0.973, 0.888, 0.945)}
PAPER_T4 = {'MLP': (5.6, 3.9, 0.4, 1.1), 'XGB': (6.2, 3.9, 0.4, 1.1), 'RF': (5.6, 3.7, 0.4, 1.3)}
OUT = P + 'Repro/'; os.makedirs(OUT, exist_ok=True)

def load_pairs(f):
    d = {}
    for l in open(f):
        t = l.rstrip('\n').split('\t'); d[(t[0].split('/')[-1], t[1].split('/')[-1])] = np.array(t[2:], float)
    return d
def dataset():
    ents, y = [], []
    for l in open(DATA):
        t = l.rstrip('\n').split('\t'); ents.append((t[0], t[1])); y.append(int(t[2]))
    return ents, np.array(y)
def folds():
    return {r: [int(l) for l in open(PART + str(r) + '.txt') if l.strip()] for r in range(1, 11)}
def make(alg):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neural_network import MLPClassifier
    import xgboost as xgb
    return {'RF': RandomForestClassifier, 'MLP': MLPClassifier, 'XGB': xgb.XGBClassifier}[alg](random_state=0)
def load_emb(f):
    # file la dict Python voi nhay don; doi sang JSON de doc nhanh va it RAM hon eval/literal_eval
    return json.loads(open(f).read().replace("'", '"'))

st = sys.argv[1]; t0 = time.time()
if st == 'kg':
    from build_kg import run_save_graph
    run_save_graph(P + 'go.owl', P + 'go_annotations.gaf', OUT + 'KG_rebuilt.gpickle', OUT + 'Labelclasses_rebuilt.pkl', 'PPI')
    A = pickle.load(open(OUT + 'KG_rebuilt.gpickle', 'rb')); B = pickle.load(open(P + 'KG.gpickle', 'rb'))
    import rdflib
    named = lambda G: set((str(u), str(v)) for u, v in G.edges() if not isinstance(u, rdflib.term.BNode) and not isinstance(v, rdflib.term.BNode))
    nb = lambda G: sum(1 for u, v in G.edges() if isinstance(u, rdflib.term.BNode) or isinstance(v, rdflib.term.BNode))
    ea, eb = named(A), named(B)
    # blank node (BNode) cua OWL nhan ID ngau nhien moi lan parse, nen chi so cac canh giua node co ten
    r = {'nodes_rebuilt': A.number_of_nodes(), 'nodes_authors': B.number_of_nodes(), 'named_edges_rebuilt': len(ea), 'named_edges_authors': len(eb),
         'named_edges_only_rebuilt': len(ea - eb), 'named_edges_only_authors': len(eb - ea), 'bnode_edges_rebuilt': nb(A), 'bnode_edges_authors': nb(B)}
elif st == 'emb':
    if sys.argv[2] == 'authors':
        r = {'emb_file': P + 'Embeddings/Emb_classes_maxdepth4_nwalks100.txt'}
    else:
        from train_embedding_model import compute_embeddings_classes
        f = OUT + 'Emb_classes_retrained.txt'
        compute_embeddings_classes(P + 'go.owl', f, 100, 100, 4)
        r = {'emb_file': f, 'n_classes': len(load_emb(f))}
elif st == 'pairs':
    # mode 'repo': goi nguyen ham run_PairEmbeddings cua repo cho moi cap (cham, khoang 1-2 gio).
    # mode 'fast': cung dinh nghia DCA, chi luu cache nx.descendants (vai phut); kem kiem tra voi ham repo tren 500 cap dau.
    import networkx as nx, rdflib
    from generate_pair_representations import run_PairEmbeddings
    emb, out = sys.argv[2], sys.argv[3]; mode = sys.argv[4] if len(sys.argv) > 4 else 'fast'; N_CHECK = 500
    AUTH = P + 'Embeddings/Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.txt'
    if mode == 'repo':
        run_PairEmbeddings(emb, out, P + 'KG.gpickle', DATA, 100)
    else:
        G = pickle.load(open(P + 'KG.gpickle', 'rb')); E = load_emb(emb); ents, _ = dataset(); desc = {}
        def Dn(n):
            if n not in desc: desc[n] = frozenset(nx.descendants(G, n)) if G.has_node(n) else frozenset()
            return desc[n]
        with open(out, 'w') as fo:
            for a, b in ents:
                com = Dn(rdflib.term.URIRef(a)) & Dn(rdflib.term.URIRef(b))
                dca = [c for c in com if not any(o != c and c in Dn(o) for o in com) and str(c) in E]
                v = np.average(np.array([E[str(c)] for c in dca]), 0).tolist() if dca else [0] * 100
                fo.write(a + '\t' + b + ''.join('\t' + str(x) for x in v) + '\n')
        sub = OUT + 'pairs_subset.txt'; open(sub, 'w').writelines(open(DATA).readlines()[:N_CHECK])
        run_PairEmbeddings(emb, OUT + 'pairs_subset_repo.txt', P + 'KG.gpickle', sub, 100)
        R, F_ = load_pairs(OUT + 'pairs_subset_repo.txt'), load_pairs(out)
        dd = max(np.abs(R[k] - F_[k]).max() for k in R)
        print(f'fast vs ham repo tren {len(R)} cap dau: lech lon nhat {dd:.3g}')
    A, B = load_pairs(out), load_pairs(AUTH)
    ks = [k for k in A if k in B]; d = np.array([np.abs(A[k] - B[k]).max() for k in ks])
    r = {'mode': mode, 'pairs': len(A), 'pairs_matched_authors': len(ks), 'max_abs_diff_vs_authors': float(d.max()),
         'frac_pairs_identical_vs_authors': float((d < 1e-9).mean())}
    if mode != 'repo': r['max_abs_diff_fast_vs_repo_function_first500'] = float(dd)
elif st == 'ml':
    from sklearn.metrics import precision_score, recall_score, f1_score
    pf, tag = sys.argv[2], sys.argv[3]; algs = sys.argv[4].split(',') if len(sys.argv) > 4 else ['RF', 'XGB', 'MLP']
    R = load_pairs(pf); ents, y = dataset(); X = np.array([R[(a, b)] for a, b in ents]); F = folds(); r = {}
    for alg in algs:
        sc = []
        for k in range(1, 11):
            te = F[k]; m = np.ones(len(y), bool); m[te] = False
            clf = make(alg).fit(X[m], y[m]); p = clf.predict(X[te])
            sc.append((precision_score(y[te], p), recall_score(y[te], p), f1_score(y[te], p, average='weighted')))
            pickle.dump(clf, open(OUT + f'Model_{tag}_{alg}_{k}.pickle', 'wb'))
        med = np.median(np.array(sc), 0).round(4).tolist()
        r[alg] = {'median_Pr_Re_F1': med, 'paper_Pr_Re_F1': PAPER_T2[alg], 'folds_F1': [round(s[2], 4) for s in sc]}
        print(alg, r[alg], flush=True)
elif st == 'expl':
    import networkx as nx, rdflib
    from sklearn.metrics import precision_score, recall_score, f1_score
    pf, ef, tag, alg = sys.argv[2:6]; mx = int(sys.argv[6]) if len(sys.argv) > 6 else None
    G = pickle.load(open(P + 'KG.gpickle', 'rb')); E = load_emb(ef); ents, y = dataset(); F = folds()
    desc = {}
    def D(n):
        if n not in desc: desc[n] = frozenset(str(x) for x in nx.descendants(G, rdflib.term.URIRef(n))) if G.has_node(rdflib.term.URIRef(n)) else frozenset()
        return desc[n]
    def dcas(a, b):  # giong het getExplanations / run_PairEmbeddings
        com = D(a) & D(b)
        return [c for c in com if not any(o != c and c in D(o) for o in com) and c in E]
    avg = lambda V: np.mean(V, 0) if len(V) else np.zeros(100)
    rows, nec_len, suf_len, dca_len = [], [], [], []
    for k in range(1, 11):
        clf = pickle.load(open(OUT + f'Model_{tag}_{alg}_{k}.pickle', 'rb'))
        # partition cua repo xep cap duong truoc cap am, nen lay mau ngau nhien (seed = fold) thay vi N cap dau
        te = sorted(np.random.RandomState(k).choice(F[k], mx, replace=False)) if mx else F[k]
        cy, cg, cw, iy, ig, io = [], [], [], [], [], []
        for i in te:
            a, b = ents[i]; ds = dcas(a, b); V = [np.array(E[c]) for c in ds]; n = len(V)
            # gom moi bien the vao mot lan goi predict: [global, without_1..n, only_1..n]
            Xs = [avg(V)] + [avg(V[:j] + V[j + 1:]) for j in range(n)] + [V[j] for j in range(n)]
            pr = clf.predict(np.array(Xs)); g = pr[0]
            nec = [j for j in range(n) if pr[1 + j] != g]; suf = [j for j in range(n) if pr[1 + n + j] == g]
            p2 = clf.predict(np.array([avg([V[j] for j in range(n) if j not in nec]), avg([V[j] for j in suf])]))
            nec_len.append(len(nec)); suf_len.append(len(suf)); dca_len.append(n)
            if g == y[i]: cy.append(y[i]); cg.append(g); cw.append(p2[0])
            else: iy.append(y[i]); ig.append(g); io.append(p2[1])
        met = lambda t, p: (precision_score(t, p, zero_division=0), recall_score(t, p, zero_division=0), f1_score(t, p, average='weighted', zero_division=0))
        a1, a2, b1, b2 = met(cy, cg), met(cy, cw), met(iy, ig), met(iy, io)
        b3 = met(ig, io)  # only sufficient, cham theo du doan goc cua mo hinh thay vi nhan that
        rows.append([a2[j] - a1[j] for j in range(3)] + [b2[j] - b1[j] for j in range(3)] + list(b3))
        print('fold', k, np.round(rows[-1], 3).tolist(), round(time.time() - t0), 's', flush=True)
    med = np.median(rows, 0).round(3).tolist()
    r = {'table3_median_dPr_dRe_dF1_wo_necessary__dPr_dRe_dF1_only_sufficient': med[:6],
         'only_sufficient_vs_model_prediction_Pr_Re_F1': med[6:],
         'paper_table3_RDF2Vec': PAPER_T3[alg],
         'table4_suff_avg_std_nec_avg_std': [round(np.mean(suf_len), 2), round(np.std(suf_len), 2), round(np.mean(nec_len), 2), round(np.std(nec_len), 2)],
         'paper_table4_RDF2Vec': PAPER_T4[alg], 'shared_aspects_avg_std': [round(np.mean(dca_len), 2), round(np.std(dca_len), 2)], 'paper_shared': (9.1, 6.5)}
r['seconds'] = round(time.time() - t0)
print(json.dumps(r, indent=1))
name = {'ml': f'ml_{sys.argv[3] if len(sys.argv) > 3 else ""}', 'expl': f'expl_{sys.argv[4] if len(sys.argv) > 4 else ""}_{sys.argv[5] if len(sys.argv) > 5 else ""}'}.get(st, st)
json.dump(r, open(OUT + 'result_' + name + '.json', 'w'), indent=1, default=float)

# Phần A. KGsim2vec
## A1. Tải repo, dữ liệu, Java

In [ ]:
if RUN_KGSIM2VEC:
    %cd /content
    !rm -rf ExplainablePPI && git clone -q https://github.com/liseda-lab/ExplainablePPI.git
    %cd /content/ExplainablePPI
    !git log -1 --format='commit %H'
    !cd Experimental_Results/Data && unzip -o -q go-basic.zip && unzip -o -q goa_human.zip && ls -la
    !pip -q install rdflib six sympy
    !java -version 2>&1 | head -1 || true
    !which javac >/dev/null || (apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless >/dev/null)
    !javac -version

## A2–A3. Chọn semantic aspect (α = 10, β = 0, γ = 0)
Gọi `ss_calculation_sas.get_semantic_aspects` của repo, rồi so với danh sách tác giả đã commit (`kgsim2vec_sas.txt`).

In [ ]:
if RUN_KGSIM2VEC:
    %cd /content/ExplainablePPI
    import sys, time; sys.path.insert(0, '.')
    import ss_calculation_sas
    t = time.time()
    names, str_aspects = ss_calculation_sas.get_semantic_aspects('Experimental_Results/Results/kgsim2vec_sas_REPRO.txt',
        'Experimental_Results/Data/go-basic.owl', 'Experimental_Results/Data/goa_human.gaf', 10, 0, 0)
    urls = [u.strip('"') for u in str_aspects.split()]
    open(f'{RES}/kg_asp_urls.txt', 'w').write('\n'.join(urls) + '\n')
    open(f'{RES}/kg_asp_names.txt', 'w').write('\n'.join(names) + '\n')
    authors = [l.split('\t')[0] for l in open('Experimental_Results/Results/kgsim2vec_sas.txt') if l.strip()]
    print(f'{len(urls)} aspect, {time.time()-t:.0f} s | trùng tập aspect của tác giả: {set(urls) == set(authors)}')
    print('Có protein-containing complex (GO_0032991):', 'http://purl.obolibrary.org/obo/GO_0032991' in urls)

## A4. Tính đặc trưng bằng SML Java gốc
Đúng lệnh của `ss_calculation_sas.main`, chỉ đổi classpath `;` thành `:`. Mất khoảng 8 phút. Kết quả được so với `Experimental_Results/Results/ss_resnik_max_ICseco.txt` của tác giả.

In [ ]:
if RUN_KGSIM2VEC:
    %cd /content/ExplainablePPI
    !javac -cp ".:./SS_Calculation/jar_files/*" ./SS_Calculation/Run_SS_calculation_SAs.java
    os.makedirs('Results_repro', exist_ok=True)
    cmd = ('java -Xmx8g -cp ".:./SS_Calculation/jar_files/*" SS_Calculation/Run_SS_calculation_SAs '
           '"Experimental_Results/Data/go-basic.owl" "Experimental_Results/Data/goa_human.gaf" "GO" "http://purl.obolibrary.org/obo/GO_" "gaf" '
           '"Experimental_Results/Data/v11(score950).txt" "Results_repro/" "resnik_max_ICseco" "' + str(len(urls)) + '" ' + str_aspects)
    t = time.time(); rc = subprocess.run(cmd + ' > sml.log 2>&1', shell=True).returncode
    print('SML xong, mã thoát', rc, f'{(time.time()-t)/60:.1f} phút')
    !python {S}/kg_compare_sml.py Results_repro/ss_resnik_max_ICseco.txt {RES}/kg_asp_urls.txt Experimental_Results/Results/ss_resnik_max_ICseco.txt Experimental_Results/Results/kgsim2vec_sas.txt

## A5. Hiệu năng dự đoán: bảng 1 và bảng 10 của Sousa24
DT6, DT, RF, XGB với tham số mặc định như `Prediction/ML.py`, 10 fold của repo, median WAF. GP và GP6x không chạy vì mỗi fold mất nhiều giờ (bài báo: 0.866 và 0.866).

In [ ]:
if RUN_KGSIM2VEC:
    %cd /content/ExplainablePPI
    !python {S}/kg_predict.py . Results_repro/ss_resnik_max_ICseco.txt {RES}/kg_asp_urls.txt {RES}/kg_prediction.json
    import pandas as pd
    paper = {'RF': 0.919, 'XGB': 0.915, 'DT': 0.899, 'DT6': 0.906}
    r = json.load(open(f'{RES}/kg_prediction.json'))
    display(pd.DataFrame([{'Cấu hình': k.split('/')[0], 'Mô hình': k.split('/')[1], 'Median WAF': v['median_WAF'], 'IQR': v['IQR'],
                           'Bài báo (51 aspect)': paper[k.split('/')[1]]} for k, v in r.items()]))

## A6. Giải thích và đánh giá size / informativeness (Sousa24, Fig. 4)
Fold 1. DT6 và DT: đường quyết định cho mọi cặp test (`ML.performance_DT`). RF và XGB: LIME3 và LIME8 (`Prediction/LIMEexplanations.py`) trên `KG_LIME_PAIRS` cặp. Đánh giá bằng `evaluation_explanations.py`: size = số aspect, informativeness = tổng và trung bình IC của aspect. Bài báo chỉ báo bằng hình, không có số, nên so về xu hướng: LIME8 dài hơn và có IC trung bình thấp hơn DT6.

In [ ]:
if RUN_KGSIM2VEC:
    %cd /content/ExplainablePPI
    !python {S}/kg_explain.py Results_repro/ss_resnik_max_ICseco.txt {RES}/kg_asp_names.txt {KG_LIME_PAIRS} {RES}/kg_explain
    print(json.dumps(json.load(open(f'{RES}/kg_explain/kg_explain_summary.json')), indent=1))

# Phần B. SEEK
## B1. Tải repo, dữ liệu, môi trường
Tạo virtualenv `/content/sv` với đúng phiên bản trong `req.txt` (numpy 2.0.0 đổi thành 1.26.4 vì gensim 4.3.2 cần numpy 1.x).

In [ ]:
if RUN_SEEK:
    %cd /content
    !rm -rf seek && git clone -q https://github.com/liseda-lab/seek.git
    %cd /content/seek
    !git log -1 --format='commit %H'
    !cd PPI && unzip -o -q go.zip && unzip -o -q go_annotations.zip && unzip -o -q KG.zip
    !cd PPI/Embeddings && unzip -o -q Emb_classes_maxdepth4_nwalks100.zip && unzip -o -q Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.zip
    !pip -q install virtualenv && virtualenv -q /content/sv
    !/content/sv/bin/pip -q install numpy==1.26.4 scipy==1.12.0 pandas==2.2.2 scikit-learn==1.5.1 gensim==4.3.2 networkx==3.2.1 rdflib==7.0.0 xgboost==2.1.0
    !cp {S}/seek_repro.py /content/seek/
    SV = '/content/sv/bin/python'

## B3. Dựng lại knowledge graph
`build_kg.run_save_graph` từ `go.owl` + `go_annotations.gaf`, rồi so số node và cạnh với `KG.gpickle` tác giả để sẵn.

In [ ]:
if RUN_SEEK and SEEK_REBUILD_KG:
    %cd /content/seek
    !{SV} seek_repro.py kg | tail -12

## B4. Embedding lớp GO bằng RDF2Vec
`authors`: dùng file embedding tác giả để sẵn, cho phép so chính xác. `retrain`: chạy lại `train_embedding_model.compute_embeddings_classes` (100 chiều, 100 walk, độ sâu 4); kết quả ngẫu nhiên nên F1 lệch nhẹ.

In [ ]:
if RUN_SEEK:
    %cd /content/seek
    !{SV} seek_repro.py emb {SEEK_EMB} | tail -5
    EMB = 'PPI/Embeddings/Emb_classes_maxdepth4_nwalks100.txt' if SEEK_EMB == 'authors' else 'PPI/Repro/Emb_classes_retrained.txt'

## B5. Biểu diễn cặp: trung bình embedding của các disjoint common ancestor
Mặc định (`fast`) dùng đúng định nghĩa DCA của `generate_pair_representations.run_PairEmbeddings` nhưng lưu cache, rồi chạy chính hàm repo trên 500 cặp đầu để chứng minh trùng khớp (lệch 0). Sau đó so cả 47,142 cặp với file của tác giả: với embedding của tác giả phải lệch 0; với embedding huấn luyện lại thì khác (do ngẫu nhiên).

In [ ]:
if RUN_SEEK:
    %cd /content/seek
    PAIRS = f'PPI/Repro/pairs_{SEEK_EMB}.txt'
    !{SV} seek_repro.py pairs {EMB} {PAIRS} {SEEK_PAIR_MODE} | grep -v '/' | tail -10

## B6. Hiệu năng dự đoán: bảng 2 của Sousa23 (RDF2Vec + SEEK)
RF, XGB, MLP mặc định như `train_ml.py`, 10 fold của repo. Báo median precision, recall (lớp dương) và weighted F1.

In [ ]:
if RUN_SEEK:
    %cd /content/seek
    !{SV} seek_repro.py ml {PAIRS} {SEEK_EMB} {','.join(SEEK_ALGS)} | tail -20

## B7. Hiệu quả và độ dài giải thích: bảng 3 và 4 của Sousa23
Với mỗi cặp test, tìm aspect *necessary* (bỏ đi thì đổi dự đoán) và *sufficient* (chỉ giữ nó vẫn ra cùng dự đoán), đúng như `generate_seek_explanation.getExplanations`. Bảng 3: bỏ mọi aspect necessary ở các cặp đoán đúng, và chỉ giữ aspect sufficient ở các cặp đoán sai.

**Lưu ý về "only sufficient".** Theo định nghĩa trong repo, aspect sufficient là aspect giữ *nguyên dự đoán gốc*. Với cặp đoán sai, dự đoán gốc đã sai, nên chấm theo nhãn thật thì ΔF1 luôn gần 0. Con số 0.945–1.000 của bài báo chỉ khớp nếu chấm theo **dự đoán gốc của mô hình** (tức đo độ trung thực). Notebook báo cả hai cách; cách hiểu này là suy luận, bài báo không nói rõ.

In [ ]:
if RUN_SEEK:
    %cd /content/seek
    mx = '' if SEEK_EXPL_MAX_PER_FOLD is None else str(SEEK_EXPL_MAX_PER_FOLD)
    for alg in SEEK_ALGS:
        print('=====', alg)
        !{SV} seek_repro.py expl {PAIRS} {EMB} {SEEK_EMB} {alg} {mx} | tail -16

# Phần C. Tổng hợp

In [ ]:
import glob, pandas as pd
rows = []
if os.path.exists(f'{RES}/kg_prediction.json'):
    paper = {'RF': 0.919, 'XGB': 0.915, 'DT': 0.899, 'DT6': 0.906}
    for k, v in json.load(open(f'{RES}/kg_prediction.json')).items():
        rows.append(['KGsim2vec ' + k.split('/')[0], k.split('/')[1], 'median WAF', v['median_WAF'], paper[k.split('/')[1]]])
for f in sorted(glob.glob('/content/seek/PPI/Repro/result_ml_*.json')):
    for alg, v in json.load(open(f)).items():
        if alg == 'seconds': continue
        rows.append(['SEEK emb=' + f.split('result_ml_')[1][:-5], alg, 'median F1', v['median_Pr_Re_F1'][2], v['paper_Pr_Re_F1'][2]])
for f in sorted(glob.glob('/content/seek/PPI/Repro/result_expl_*.json')):
    v = json.load(open(f)); alg = f.split('_')[-1][:-5]
    t3 = v['table3_median_dPr_dRe_dF1_wo_necessary__dPr_dRe_dF1_only_sufficient']; p3 = v['paper_table3_RDF2Vec']
    t4 = v['table4_suff_avg_std_nec_avg_std']; p4 = v['paper_table4_RDF2Vec']
    rows += [['SEEK', alg, 'ΔF1 w/o necessary', t3[2], p3[2]], ['SEEK', alg, 'ΔF1 only sufficient (so với nhãn thật)', t3[5], p3[5]],
             ['SEEK', alg, 'F1 only sufficient (so với dự đoán gốc)', v['only_sufficient_vs_model_prediction_Pr_Re_F1'][2], p3[5]],
             ['SEEK', alg, 'độ dài sufficient', t4[0], p4[0]], ['SEEK', alg, 'độ dài necessary', t4[2], p4[2]],
             ['SEEK', alg, 'số aspect chung', v['shared_aspects_avg_std'][0], 9.1]]
df = pd.DataFrame(rows, columns=['Pipeline', 'Mô hình', 'Chỉ số', 'Tái hiện', 'Bài báo'])
df['Chênh lệch'] = (df['Tái hiện'] - df['Bài báo']).round(3)
display(df); df.to_csv(f'{RES}/tong_hop_so_voi_bai_bao.csv', index=False)
!cp /content/seek/PPI/Repro/*.json {RES}/ 2>/dev/null; cd /content && zip -q -r repro_results.zip repro_results && ls -la repro_results.zip
try:
    from google.colab import files; files.download('/content/repro_results.zip')
except Exception as e:
    print('Tải thủ công /content/repro_results.zip', e)